In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os
from collections import defaultdict
import matplotlib.pylab as pl


In [ ]:
output_folder=("../PhysiCell/results/stochastic_cell_cycle/cell_volumes.csv")
pc_df = pd.read_csv(output_folder,float_precision='round_trip').sort_values(by=['dt']).reset_index(drop=True)
pc_df


In [ ]:
# Normalize time to hours
pc_df['dt'] = pc_df['dt'] / 60
pc_init_vol = 523.6

# Compute average volume per time step (normalized)
pc_avg_vol = pc_df.groupby('dt')['total_volume'].mean()
pc_avg_vol

In [ ]:
pc_dts = pc_df['dt'].unique()
pc_vols = (pc_avg_vol / pc_init_vol * 100).tolist()
pc_vols

In [ ]:
pc_init_vol = pc_df.iloc[0,4]
pc_group =  pc_df.groupby(['id']).groups.items()
num_cels = len(pc_group)
colors = pl.cm.Greens(np.linspace(0,1,num_cels*2))
j=2
for cell_id,idx in pc_group:
    vols = []
    dts = []
    for i in idx.values:
        vols.append((pc_df.loc[i,'total_volume']/pc_init_vol)*100)
        dts.append(pc_df.loc[i,'dt'])
    # plt.plot(dts,vols,label="PhysiCell", color=colors[j] ,linewidth=2)
    if cell_id!=num_cels-1:
        plt.plot(dts,vols, color=colors[j] ,linewidth=2)
    else:
        plt.plot(dts,vols, color=colors[j],label = 'PhysiCell' ,linewidth=2)
    j+=1

In [ ]:
output_folder=("../Biodynamo/unit_test_cellcycle_stoch/new_results/output/")
cell_indices = [0, 2, 3, 4]
all_dfs = []

for idx in cell_indices:
    fname = os.path.join(output_folder, f'cell-{idx}.csv')
    df = pd.read_csv(fname, names=['timestep', 'volume', 'Phase', 'Age'], header=None)
    df['cell_id'] = idx
    all_dfs.append(df)

# Concatenate all individual DataFrames into one
combined_df = pd.concat(all_dfs, ignore_index=True)
grouped = combined_df.groupby('timestep').agg(
    total_volume=('volume', 'sum'),
    n_cells=('cell_id', 'nunique')  # number of unique cells at each timestep
)

# Calculate average volume per cell
grouped['avg_volume_per_cell'] = grouped['total_volume'] / grouped['n_cells']
grouped['avg_volume_per_cell'] = grouped['avg_volume_per_cell'] / 523.599

# Reset index to get 'timestep' as a column
bd_df = grouped.reset_index()

combined_df

In [ ]:
output_folder=("../Chaste/unit_test_cellcycle_stoch/results/cellcycle_stochastic.dat")
ch_df = pd.DataFrame(columns=["dt","id","x","y","z","g1_duration","s_duration","g2_duration","m_duration","current_phase","target_area","volume"])
with open(output_folder, "r") as f:
    for line in f.read().splitlines():
        data = line.split()
        data = [float(x) if x.replace('.', '', 1).isdigit() else x for x in data]
        dt = data[0]
        for i in range(0,len(data)//11):
            row = {'dt': dt, 'id': data[1+11*i], 'x': data[2+11*i], 'y': data[3+11*i], 'z': data[4+11*i], 'g1_duration': data[5+11*i], 's_duration': data[6+11*i], 'g2_duration': data[7+11*i], 'm_duration': data[8+11*i], 'current_phase': data[9+11*i], 'target_area': data[10+11*i], 'volume': data[11+11*i]}
            row = pd.Series(row)
            df2 = pd.DataFrame(row).transpose()

            ch_df =pd.concat([ch_df,df2],ignore_index=True)
ch_df

In [ ]:
output_folder=("../Tisim/unit_test_cellcycle_stoch/cell cycle stochastic.csv")
data = pd.read_csv(output_folder, header=0)
time_steps = data['time (hour)']
volume_columns = data.columns[1:]  # Assuming all other columns are cell volumes

average_volumes = data[volume_columns].mean(axis=1, skipna=True)
initial_total_volume = average_volumes.iloc[0]

percentage_total_volumes = (average_volumes / initial_total_volume)*100
# print(percentage_total_volumes)
# Create a new DataFrame with time steps and percentage of total volumes
ts_df = pd.DataFrame({
    'timestep': time_steps,
    'volumes': percentage_total_volumes
})
ts_df



In [ ]:
fig, ax = plt.subplots()



# Plot PhysiCell
ax.plot(pc_dts, pc_avg_vol, label="PhysiCell", color='green', linewidth=2)

# # Chaste
# ch_init_vol = ch_df.loc[0, "volume"]
# ch_avg_vol = ch_df.groupby('dt')['volume'].mean()
# dts = ch_avg_vol.index
# vols = (ch_avg_vol / ch_init_vol * 100).tolist()
# ax.plot(dts, vols, label="Chaste", color='blue', linewidth=2, alpha=0.5)

# # TiSim
# ax.plot(ts_df['timestep'], ts_df['volumes'], label="TiSim", color='#ffd343', linewidth=2)

# # BioDynaMo
# ax.plot(bd_df['timestep'], bd_df["avg_volume_per_cell"], label="Biodynamo", color="red")

# Add phase background rectangles (G0/G1, S, G2, M)
for i in range(3):  # repeat 3 cell cycles
    base = 18 * i
    phases = [
        (base, 7, 'orange', 'G0/G1'),
        (base + 7, 6, 'purple', 'S'),
        (base + 13, 3, 'yellow', 'G2'),
        (base + 16, 2, 'red', 'M'),
    ]
    for x, w, color, label in phases:
        ax.add_patch(patches.Rectangle((x, 50), w, 170, facecolor=color, alpha=0.1, edgecolor=None))
        if label in ['G2', 'M']:
            ax.text(x + w * 0.3, 210, label, color=color, alpha=0.3)
        else:
            ax.text(x + w * 0.3, 210, label, color=color, alpha=0.3)

# Final plot formatting
ax.set_xlim(0, 48)
ax.set_ylim(90)
ax.set_xlabel("Time (hours)", fontsize=12, color='#262626')
ax.set_ylabel("Percentage of initial volume", fontsize=12, color='#262626')
ax.set_title('Fixed Cell Cycle Total Volume', color='#262626')
ax.legend(bbox_to_anchor=(1.0, 1.0), loc='upper left')
plt.tight_layout()
plt.savefig("stochastic.png", dpi=200)
plt.show()